In [ ]:
!nvidia-smi

Tue Oct 27 00:04:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Install (free Colab). May take several minutes.
!pip install -q --upgrade pip
!pip install -q torch transformers accelerate
!pip install -q vllm pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.2.2 which is incompatible.


In [ ]:
!pip install -q vllm --no-build-isolation

In [4]:
import time
import pandas as pd

# Small model that fits free T4 (about 16GB)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# Fixed prompts so runs are comparable
PROMPTS = [
    "Explain what batching means in LLM inference in two sentences.",
    "List three ways to reduce latency when serving language models.",
    "What is a KV cache and why does it matter for decoding?",
]

MAX_TOKENS = 64

Hugging Face baseline

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Loading HF model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
hf_model.eval()

def hf_generate(prompt, max_new_tokens=MAX_TOKENS):
    inputs = tokenizer(prompt, return_tensors="pt").to(hf_model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = hf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    new_tokens = out.shape[1] - inputs["input_ids"].shape[1]
    elapsed = t1 - t0
    return {
        "backend": "huggingface",
        "batch_size": 1,
        "new_tokens": int(new_tokens),
        "latency_s": elapsed,
        "tokens_per_s": (new_tokens / elapsed) if elapsed > 0 else 0.0,
    }

# Warm-up (ignore result)
_ = hf_generate(PROMPTS[0])

hf_rows = []
for p in PROMPTS:
    hf_rows.append(hf_generate(p))

hf_df = pd.DataFrame(hf_rows)
print(hf_df)
print("HF mean tokens/s:", hf_df["tokens_per_s"].mean())

Loading HF model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

       backend  batch_size  new_tokens  latency_s  tokens_per_s
0  huggingface           1          64   4.801767     13.328426
1  huggingface           1          64   3.129298     20.451870
2  huggingface           1          64   2.611535     24.506664
HF mean tokens/s: 19.42898650940266


vLLM benchmark

In [6]:
from vllm import LLM, SamplingParams

print("Loading vLLM model (first time can take a few minutes)...")
llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    trust_remote_code=True,
    gpu_memory_utilization=0.85,
    max_model_len=1024,
)

sampling = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_TOKENS,
)

def vllm_run(prompts, batch_label):
    # Warm-up
    _ = llm.generate(prompts[:1], sampling)
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    outputs = llm.generate(prompts, sampling)
    torch.cuda.synchronize()
    t1 = time.perf_counter()

    total_new = 0
    for o in outputs:
        total_new += len(o.outputs[0].token_ids)

    elapsed = t1 - t0
    return {
        "backend": "vllm",
        "batch_size": batch_label,
        "num_prompts": len(prompts),
        "new_tokens_total": int(total_new),
        "wall_time_s": elapsed,
        "tokens_per_s_total": (total_new / elapsed) if elapsed > 0 else 0.0,
        "latency_per_prompt_s": elapsed / max(len(prompts), 1),
    }

rows = []

# Batch size 1: one prompt at a time
for p in PROMPTS:
    rows.append(vllm_run([p], batch_label=1))

# Batch = all prompts together
rows.append(vllm_run(PROMPTS, batch_label=len(PROMPTS)))

vllm_df = pd.DataFrame(rows)
print(vllm_df)
print("vLLM batch-1 mean tokens/s:", vllm_df[vllm_df["batch_size"] == 1]["tokens_per_s_total"].mean())
print("vLLM batched tokens/s:", vllm_df[vllm_df["batch_size"] > 1]["tokens_per_s_total"].values)=

Loading vLLM model (first time can take a few minutes)...
(MainProcess pid=1556) INFO 10-27 00:13:40 [api_utils.py:286] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 1024, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}
(MainProcess pid=1556) INFO 10-27 00:14:03 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=1556) WARNING 10-27 00:14:03 [model.py:2498] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=1556) INFO 10-27 00:14:03 [model.py:2152] Using max model len 1024
(MainProcess pid=1556) INFO 10-27 00:14:03 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=1556) INFO 10-27 00:14:04 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

(MainProcess pid=1556) WARNING 10-27 00:14:08 [system_utils.py:153] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(MainProcess pid=1556) INFO 10-27 00:16:00 [hf.py:648] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

...

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.60it/s, est. speed input: 33.95 toks/s, output: 167.12 toks/s]


Rendering prompts:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 3/3 [00:00<00:00,  7.60it/s, est. speed input: 94.09 toks/s, output: 488.25 toks/s]

  backend  batch_size  num_prompts  new_tokens_total  wall_time_s  \
0    vllm           1            1                64     0.436648   
1    vllm           1            1                64     0.418338   
2    vllm           1            1                64     0.407882   
3    vllm           3            3               192     0.428150   

   tokens_per_s_total  latency_per_prompt_s  
0          146.571307              0.436648  
1          152.986398              0.418338  
2          156.907952              0.407882  
3          448.440496              0.142717  
vLLM batch-1 mean tokens/s: 152.1552188776199
vLLM batched tokens/s: [448.44049562]


In [7]:
summary = {
    "hf_mean_tokens_per_s": float(hf_df["tokens_per_s"].mean()),
    "vllm_batch1_mean_tokens_per_s": float(
        vllm_df[vllm_df["batch_size"] == 1]["tokens_per_s_total"].mean()
    ),
    "vllm_batched_tokens_per_s": float(
        vllm_df[vllm_df["batch_size"] > 1]["tokens_per_s_total"].iloc[0]
    ),
    "model": MODEL_ID,
    "gpu": "Tesla T4 (Colab free)",
    "max_new_tokens": MAX_TOKENS,
}

summary_df = pd.DataFrame([summary])
print(summary_df)

# Downloadable CSV
summary_df.to_csv("summary.csv", index=False)
hf_df.to_csv("hf_raw.csv", index=False)
vllm_df.to_csv("vllm_raw.csv", index=False)
print("Saved summary.csv, hf_raw.csv, vllm_raw.csv")

   hf_mean_tokens_per_s  vllm_batch1_mean_tokens_per_s  \
0             19.428987                     152.155219   

   vllm_batched_tokens_per_s                       model  \
0                 448.440496  Qwen/Qwen2.5-0.5B-Instruct   

                     gpu  max_new_tokens  
0  Tesla T4 (Colab free)              64  
Saved summary.csv, hf_raw.csv, vllm_raw.csv
